# 📊 Hệ Thống RAG Hỏi Đáp Báo Cáo Tài Chính (Financial Multimodal RAG)

**Môi trường thực thi:** Google Colab - GPU T4 (16 GB VRAM)  
**Model LLM:** `Qwen/Qwen2.5-3B-Instruct` (fp16)  
**Embedding & Index:** `BAAI/bge-m3` + FAISS + SQLite FTS5 (BM25)  
**Cơ sở dữ liệu số liệu có cấu trúc:** SQLite (`company`, `report`, `metric`, `fact`, `chunk`, `chunk_fts`)  
**Giao diện Demo:** Gradio Web UI (`share=True` tạo link public)  

---
### Quy trình thực hiện:
1. Kiểm tra phần cứng GPU T4.
2. Cài đặt các thư viện cần thiết (PyTorch, Transformers, Sentence-Transformers, FAISS, Gradio, PyMuPDF, pdfplumber).
3. Nạp dữ liệu và khởi tạo database số liệu (`finrag seed` & `finrag ingest`).
4. Chạy đánh giá bộ câu hỏi chuẩn (`finrag eval`).
5. Khởi chạy Gradio Web UI với Qwen2.5-3B trực tiếp trên GPU T4.

## 1. Kiểm tra GPU (GPU Check)

In [ ]:
!nvidia-smi

## 2. Cài đặt dependencies trên Colab

In [ ]:
# Cài đặt các thư viện cần thiết cho GPU T4
!pip install -q transformers accelerate sentence-transformers faiss-gpu gradio pymupdf pdfplumber typer rich pydantic pydantic-settings python-dotenv

## 3. Khởi tạo Cơ sở dữ liệu và Nạp báo cáo (Ingestion)

In [ ]:
%cd /content/financial-rag
# Khởi tạo database và từ điển chỉ tiêu Việt - Anh
!python -m financial_rag.cli seed

# Parse báo cáo PDF, chunk văn bản, trích xuất facts có cấu trúc và tạo FAISS Index
!python -m financial_rag.cli ingest

## 4. Chạy Đánh Giá Benchmark (Evaluation)

In [ ]:
# Đánh giá trên 36 câu hỏi chuẩn (Tra cứu, Tính toán, Giải thích, So sánh, Câu bẫy)
!python -m financial_rag.cli evaluate

## 5. Khởi chạy Giao diện Gradio với Qwen2.5-3B-Instruct (fp16 trên GPU T4)

In [ ]:
import os
os.environ["LLM_BACKEND"] = "qwen_local"
os.environ["LLM_MODEL_ID"] = "Qwen/Qwen2.5-3B-Instruct"
os.environ["LLM_DEVICE"] = "cuda"
os.environ["LLM_FP16"] = "true"
os.environ["EMBEDDING_MODEL"] = "BAAI/bge-m3"
os.environ["EMBEDDING_DEVICE"] = "cuda"

from financial_rag.config import settings
from financial_rag.db.repository import FinancialRepository
from financial_rag.reasoning.engine import FinancialRAGEngine
from financial_rag.llm.model import LLMBackend
from financial_rag.ui.app import create_gradio_app

db_p = settings.resolve_path(settings.database_path)
repo = FinancialRepository(db_p)
llm = LLMBackend(backend_type="qwen_local", model_id="Qwen/Qwen2.5-3B-Instruct", device="cuda")
engine = FinancialRAGEngine(repo=repo, llm=llm)

# Khởi tạo Gradio app với public link share=True
app = create_gradio_app(engine=engine, repo=repo)
app.launch(share=True, debug=True)